In [1]:
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score


In [3]:
df = pd.read_csv("uber.csv")

In [4]:
df.head()

,Unnamed: 0,key,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
0,24238194,2015-05-07 19:52:06.0000003,7.5,2015-05-07 19:52:06 UTC,-73.999817,40.738354,-73.999512,40.723217,1
1,27835199,2009-07-17 20:04:56.0000002,7.7,2009-07-17 20:04:56 UTC,-73.994355,40.728225,-73.994710,40.750325,1
2,44984355,2009-08-24 21:45:00.00000061,12.9,2009-08-24 21:45:00 UTC,-74.005043,40.740770,-73.962565,40.772647,1
3,25894730,2009-06-26 08:22:21.0000001,5.3,2009-06-26 08:22:21 UTC,-73.976124,40.790844,-73.965316,40.803349,3
4,17610152,2014-08-28 17:47:00.000000188,16.0,2014-08-28 17:47:00 UTC,-73.925023,40.744085,-73.973082,40.761247,5


In [16]:
print(df.isnull().sum())

Unnamed: 0           0
fare_amount          0
pickup_longitude     0
pickup_latitude      0
dropoff_longitude    0
dropoff_latitude     0
passenger_count      0
hour                 0
day                  0
month                0
dtype: int64


In [18]:
df.replace([np.inf, -np.inf], np.nan, inplace=True)

In [5]:
# preprocessing
df.dropna(inplace=True) # it will drop missing values

In [19]:
df = df[
    (df['pickup_latitude'].between(-90, 90)) &
    (df['dropoff_latitude'].between(-90, 90)) &
    (df['pickup_longitude'].between(-180, 180)) &
    (df['dropoff_longitude'].between(-180, 180))
]

In [6]:
# Convert datetime
df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'])

In [7]:
# it will extract useful features 
df['hour'] = df['pickup_datetime'].dt.hour
df['day'] = df['pickup_datetime'].dt.day
df['month'] = df['pickup_datetime'].dt.month

In [20]:
df['distance'] = np.sqrt(
    (df['dropoff_longitude'] - df['pickup_longitude'])**2 +
    (df['dropoff_latitude'] - df['pickup_latitude'])**2
)

In [8]:
df.drop(['pickup_datetime', 'key'], axis=1, inplace=True) #it will drop unnecessary columns 

In [9]:
# outlier removal
df = df[(df['fare_amount'] > 0) & (df['fare_amount'] < 100)] # remove fare values

In [11]:
# correlation
print("\nCorrelation Matrix:\n")
print(df.corr())


Correlation Matrix:

                   Unnamed: 0  fare_amount  pickup_longitude  pickup_latitude  \
Unnamed: 0           1.000000     0.000072          0.000258        -0.000361   
fare_amount          0.000072     1.000000          0.007310        -0.005949   
pickup_longitude     0.000258     0.007310          1.000000        -0.816241   
pickup_latitude     -0.000361    -0.005949         -0.816241         1.000000   
dropoff_longitude    0.000349     0.006477          0.832936        -0.774655   
dropoff_latitude     0.000190    -0.008515         -0.846251         0.702137   
passenger_count      0.002239     0.012334         -0.000505        -0.001487   
hour                 0.000164    -0.020717          0.002689        -0.004034   
day                  0.000564     0.001659          0.005174        -0.008257   
month                0.001368     0.024098         -0.004701         0.004654   

                   dropoff_longitude  dropoff_latitude  passenger_count  \
Unnamed: 0 

In [12]:
# feature and target values
X = df.drop("fare_amount", axis=1)
y = df["fare_amount"]


In [13]:
# train and test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)


In [17]:
print(X_train.isnull().sum())
print(X_test.isnull().sum())
print(y_train.isnull().sum())
print(y_test.isnull().sum())

Unnamed: 0           0
pickup_longitude     0
pickup_latitude      0
dropoff_longitude    0
dropoff_latitude     0
passenger_count      0
hour                 0
day                  0
month                0
dtype: int64
Unnamed: 0           0
pickup_longitude     0
pickup_latitude      0
dropoff_longitude    0
dropoff_latitude     0
passenger_count      0
hour                 0
day                  0
month                0
dtype: int64
0
0


In [14]:
# model
# Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)

LinearRegression()

In [23]:
# Random Forest
rf = RandomForestRegressor(
    n_estimators=20,     
    max_depth=10,        
    n_jobs=-1,
    random_state=42
)
rf.fit(X_train, y_train)

RandomForestRegressor(max_depth=10, n_estimators=20, n_jobs=-1, random_state=42)

In [24]:
# evaluation 
lr_pred = lr.predict(X_test)
rf_pred = rf.predict(X_test)
print("\nLinear Regression RMSE:", np.sqrt(mean_squared_error(y_test, lr_pred)))
print("Linear Regression R2:", r2_score(y_test, lr_pred))
print("\nRandom Forest RMSE:", np.sqrt(mean_squared_error(y_test, rf_pred)))
print("Random Forest R2:", r2_score(y_test, rf_pred))



Linear Regression RMSE: 9.337585643555585
Linear Regression R2: 0.0011096110491072286

Random Forest RMSE: 4.797251719987526
Random Forest R2: 0.736346432196807


In [25]:
# for pickle file
pickle.dump(rf, open("model.pkl", "wb"))
print("\nModel saved as model.pkl")


Model saved as model.pkl
